In [ ]:
from pyspark.sql.functions import lit, col

In [ ]:
dbutils.widgets.text("catalog", "olist_project_dev")

dbutils.widgets.text("bronze_schema", "olist_bronze")
dbutils.widgets.text("raw_olist_orders_items_table", "olist_orders_items")

dbutils.widgets.text("silver_schema", "olist_silver")
dbutils.widgets.text("orders_items_table", "orders_items_silver")

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_orders_items_table_name = dbutils.widgets.get("raw_olist_orders_items_table")

silver_schema = dbutils.widgets.get("silver_schema")
orders_items_table_name = dbutils.widgets.get("orders_items_table")

In [ ]:
raw_olist_orders_items_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_orders_items_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{raw_olist_orders_items_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{raw_olist_orders_items_table_name} (
            orderId STRING,
            orderItemId INT,
            productId STRING,
            sellerId STRING,
            shippingLimitDate TIMESTAMP,
            price DOUBLE,
            freightValue DOUBLE
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
orders_items_silver_df = (
    raw_olist_orders_items_df
    .where(
        (col("order_id").rlike("^[0-9a-fA-F]{32}$")) & 
        (col("product_id").rlike("^[0-9a-fA-F]{32}$")) &
        (col("seller_id").rlike("^[0-9a-fA-F]{32}$"))
    )
    .select(
        col("order_id").alias("orderId"),
        col("order_item_id").alias("orderItemId"),
        col("product_id").alias("productId"),
        col("seller_id").alias("sellerId"),
        col("shipping_limit_date").alias("shippingLimitDate"),
        col("price").alias("price"),
        col("freight_value").alias("freightValue")
    )
)

In [ ]:
orders_items_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.{orders_items_table_name}")